# 01 · Full-volume CT/MR feature similarity

Select a reference point and compare ISO, REL and SA across the complete scan. The overlays show cosine similarity to the reference patch on uncropped axial images.


In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt

start = Path(os.environ.get("REFORM3D_REPO_ROOT", ".")).resolve()
root = next(p for p in (start, *start.parents) if (p / "notebooks/visualization_utils.py").is_file())
sys.path.insert(0, str(root / "notebooks"))
from visualization_utils import setup

config = setup(workflow="similarity")

## Load full-volume features

Set the four cases, reference points and output directory in the `similarity` section of `visualization_config.json`. The first run extracts features; later runs reload the cache. The `pca` section of the same file configures the crop/PCA examples.


In [ ]:
from visualization_utils import plot_full_volume_similarity
from reform3d.full_volume_features import load_or_extract_full_volume_features

arrays, receipt = load_or_extract_full_volume_features(config)

## MR and CT

Each row shows the complete source slice, then ISO, REL and SA. White markers identify the reference point and patch footprint; cyan contours show ground truth.


In [ ]:
num_slices = 5  # Choose 1–5 axial slices around the reference.

for case_id in ("MR", "CT"):
    plot_full_volume_similarity(
        config, arrays, receipt, case_id=case_id, num_slices=num_slices,
    )
    plt.show()


## Additional examples

KiTS23 `case_00001` adds CT with finer native in-plane spacing. BraTS FLAIR `00003` adds MR at the same standardized 1 mm resolution. Both cases and reference points were selected from source images before viewing encoder features.


In [ ]:
additional_num_slices = 5  # Choose 1–5 slices for these examples.

for case_id in ("MR-additional", "CT-additional"):
    plot_full_volume_similarity(
        config, arrays, receipt, case_id=case_id,
        num_slices=additional_num_slices,
    )
    plt.show()


## Adjust the view

Change `reference_ras_mm` in the JSON, or pass it to the plotter. Use `slice_indices=[...]` to choose up to five source axial indices. Both controls reuse cached features.

The background retains the full native frame; the shared model input is resampled to at most 224 voxels along its longest axis. Overlays use the coarser 3D patch grid and a fixed cosine scale **[−1, 1]**, with opacity proportional to positive similarity. The reference patch matches itself by definition; contours only provide anatomical context.

[Method and display options](../docs/visualization.md#full-volume-reference-feature-similarity). PCA uses its separate shared-crop workflow.
